# Soccer Data Cleaning and Schema Design

This notebook cleans the soccer datasets and documents the transactional and dimensional schemas.

In [ ]:
import pandas as pd
from pathlib import Path

# Standardizing Location & Team Names
DATA_DIR = Path("./")
CUTOFF_DATE = pd.to_datetime("2010-12-31")
END_DATE = pd.to_datetime("2026-01-01")

FILES = {
    "results": "results.csv",
    "goalscorers": "goalscorers.csv",
    "shootouts": "shootouts.csv",
    "former_names": "former_names.csv",
}

TEAM_NAME_MAP = {
    "USA": "United States",
    "U.S.A.": "United States",
    "United States of America": "United States",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "IR Iran": "Iran",
    "Korea Republic": "South Korea",
    "Republic of Korea": "South Korea",
    "Korea DPR": "North Korea",
    "PR China": "China",
    "Republic of Ireland": "Ireland",
    "West Germany": "Germany",
    "East Germany": "Germany",
    "Soviet Union": "Russia",
    "Yugoslavia": "Serbia",
    "Czech Republic": "Czechia",
    "Swaziland": "Eswatini",
    "Macedonia": "North Macedonia",
}

LOCATION_MAP = {
    "USA": "United States",
    "U.S.A.": "United States",
    "United States of America": "United States",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Korea Republic": "South Korea",
    "Republic of Korea": "South Korea",
    "PR China": "China",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Northern Ireland": "United Kingdom",
}

REQUIRED_COLS = {
    "results": ["date", "home_team", "away_team", "home_score", "away_score", "country", "city"],
    "goalscorers": ["date", "home_team", "away_team", "team", "scorer"],
    "shootouts": ["date", "home_team", "away_team", "winner"],
    "former_names": ["former", "current"],
}

# Configuring Mapping
def standardize_text(value, mapping):
    if pd.isna(value):
        return value
    value = str(value).strip()
    return mapping.get(value, value)

def convert_datetime_columns(df, dataset_name):
    if dataset_name in ["results", "goalscorers", "shootouts"] and "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    if dataset_name == "former_names":
        if "start_date" in df.columns:
            df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
        if "end_date" in df.columns:
            df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")

    return df

def add_date_parts(df, col_name="date"):
    if col_name in df.columns and pd.api.types.is_datetime64_any_dtype(df[col_name]):
        df[f"{col_name}_year"] = df[col_name].dt.year
        df[f"{col_name}_month"] = df[col_name].dt.month
        df[f"{col_name}_day"] = df[col_name].dt.day
    return df

def clean_team_columns(df):
    team_cols = ["home_team", "away_team", "team", "winner", "loser", "former", "current"]
    for col in team_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: standardize_text(x, TEAM_NAME_MAP))
    return df

def clean_location_columns(df):
    if "country" in df.columns:
        df["country"] = df["country"].apply(lambda x: standardize_text(x, LOCATION_MAP))

    if "city" in df.columns:
        df["city"] = df["city"].astype("string").str.strip()
        df["city"] = df["city"].replace({"New York": "New York City"})

    return df

def try_split_location(df):
    for col in ["location", "match_location", "venue"]:
        if col in df.columns and "city" not in df.columns and "country" not in df.columns:
            split_data = df[col].astype("string").str.split(",", n=1, expand=True)
            if split_data.shape[1] == 2:
                df["city"] = split_data[0].str.strip()
                df["country"] = split_data[1].str.strip()
    return df

def filter_by_date(df, dataset_name):
    if dataset_name in ["results", "goalscorers", "shootouts"] and "date" in df.columns:
        before = len(df)
        df = df[(df["date"] > CUTOFF_DATE) & (df["date"] < END_DATE)]
        print(f"{dataset_name}: kept {len(df)} of {before} rows after {CUTOFF_DATE.date()}.")

    elif dataset_name == "former_names":
        before = len(df)

        # Keep rows where a name was still active after cutoff
        if "end_date" in df.columns:
            df = df[
                (df["end_date"].isna()) |
                ((df["end_date"] > CUTOFF_DATE) & (df["end_date"] < END_DATE))
            ]

        print(f"{dataset_name}: kept {len(df)} of {before} rows after former-name filtering.")

    return df

def drop_missing_required_rows(df, dataset_name):
    required = [col for col in REQUIRED_COLS.get(dataset_name, []) if col in df.columns]
    before = len(df)
    df = df.dropna(subset=required)
    print(f"{dataset_name}: dropped {before - len(df)} rows with missing required values.")
    return df

def report_summary(df, dataset_name):
    print(f"\n{dataset_name.upper()} SUMMARY")
    print(f"Shape: {df.shape}")
    print("Data types:")
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())

# Load and Clean Data
dfs = {}

print("\n=== LOAD AND CLEAN PIPELINE ===")

for name, filename in FILES.items():
    print(f"\n--- Processing {name} ---")
    path = DATA_DIR / filename

    # Load
    df = pd.read_csv(path)
    print(f"Loaded: {filename}")
    print(f"Original shape: {df.shape}")

    # Convert dates
    df = convert_datetime_columns(df, name)

    # Standardize text fields
    df = clean_team_columns(df)
    df = try_split_location(df)
    df = clean_location_columns(df)

    # Filter by date (We only want 2011-2025)
    df = filter_by_date(df, name)

    # Add date parts for main event tables
    if name in ["results", "goalscorers", "shootouts"]:
        df = add_date_parts(df, "date")

    # Remove incomplete rows
    df = drop_missing_required_rows(df, name)

    # Remove duplicates
    before_dupes = len(df)
    df = df.drop_duplicates()
    print(f"{name}: removed {before_dupes - len(df)} duplicate rows.")

    # Store cleaned dataframe
    dfs[name] = df

    # Final summary
    report_summary(df, name)

# Save Clean Files
print("\n=== SAVING CLEANED FILES ===")

for name, df in dfs.items():
    output_path = DATA_DIR / f"{name}_cleaned.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

print("\n✅ Done.")


## Transactional Entity-Relationship Diagram

```mermaid
erDiagram
    TEAM {
        string team_name PK
    }

    TOURNAMENT {
        string tournament_name PK
    }

    LOCATION {
        string city
        string country
    }

    MATCH {
        int match_id PK
        string date
        string tournament_name FK
        string city FK
        string country FK
        boolean neutral_flag
    }

    MATCH_RESULT {
        int result_id PK
        int match_id FK
        string home_team FK
        string away_team FK
        int home_score
        int away_score
    }

    GOALSCORER_EVENT {
        int goal_id PK
        int match_id FK
        string team FK
        string scorer
        int minute
    }

    SHOOTOUT {
        int shootout_id PK
        int match_id FK
        string winner FK
    }
```


## Dimensional Entity-Relationship Diagram

```mermaid
erDiagram
    FACT_MATCH_RESULT {
        int match_key PK
        int date_key FK
        int home_team_key FK
        int away_team_key FK
        int tournament_key FK
        int location_key FK
        int home_score
        int away_score
        int total_goals
        int goal_diff
        boolean home_win
        boolean away_win
        boolean draw
        boolean neutral
    }

    DIM_TEAM {
        int team_key PK
        string team_name
    }

    DIM_DATE {
        int date_key PK
        int year
        int month
        int day
    }

    DIM_TOURNAMENT {
        int tournament_key PK
        string tournament_name
        string tournament_type
    }

    DIM_LOCATION {
        int location_key PK
        string city
        string country
    }
```
